# Lecture 3.4 — Agents as Tools: `Agent.as_tool()` and Manager-Style Orchestration

**Section 03 — Tools: Extending Agent Capabilities**

In Lectures 3.1 through 3.3 we built function tools, learned how the SDK generates JSON schemas from type annotations, and explored the hosted tools OpenAI provides out of the box. This lecture introduces a fundamentally different idea: making one agent callable as a tool by another agent.

`Agent.as_tool()` is the mechanism behind **manager-style orchestration** — a pattern where a central orchestrator agent calls specialist sub-agents as tools, collects their outputs, and synthesises a final result. The orchestrator never hands over control; it stays in charge throughout.

By the end of this notebook you will be able to:
- Explain the two fundamental differences between `as_tool()` and handoffs
- Register sub-agents as tools on an orchestrator and run a live example
- Inspect the `FunctionTool` that `as_tool()` returns and understand its default input schema
- Control nested run depth with `max_turns`
- Intercept and reshape sub-agent output with `custom_output_extractor`
- Replace the default string input with a structured Pydantic schema using `parameters=`

## Cell 1 — Installing the SDK

This notebook uses the `openai-agents` package pinned to a specific version. Pinning ensures every code example runs exactly as written. The SDK evolves quickly, and a later release may change default behaviours.

To install the latest version instead, remove the `==0.17.4` constraint:
```bash
pip install openai-agents
```
Or substitute any version you prefer. If the package is already present in this Colab session at the correct version, the cell completes instantly.

In [ ]:
# Pinned for reproducibility. To use the latest version,
# run: pip install openai-agents
# Or substitute your preferred version below.
!pip install openai-agents==0.17.7 openai==2.44.0 -q

## Cell 2 — API Key Setup

This notebook loads your OpenAI API key from **Google Colab Secrets**. Your key is never hardcoded or printed anywhere in this notebook.

**To add your key in Colab:**
1. Click the **key icon (🔑)** in the left sidebar.
2. Click **"+ Add new secret"**.
3. Set the **Name** to `OPENAI_API_KEY`.
4. Paste your OpenAI API key as the **Value**.
5. Toggle **"Notebook access"** to ON.
6. Run the cell below.

**Running locally?** Set the environment variable in your terminal before launching Jupyter:
```bash
export OPENAI_API_KEY="sk-..."
```

In [ ]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## Cell 3 — Model Name

Every `Agent` definition in this notebook uses the `MODEL_NAME` variable declared here. The model string is never hardcoded directly in an `Agent` — changing this one variable updates the model used across the entire notebook.

The default is `gpt-5.4-mini`, the current SDK default as of version 0.16+. For a list of available models, see the link in the comment.

In [ ]:
# See latest models at: https://platform.openai.com/docs/models
MODEL_NAME = "gpt-5.4-mini"

## Cell 4 — Imports

This cell imports everything needed across all five hands-on cells.

| Import | Source | Why we need it |
|---|---|---|
| `Agent`, `ModelSettings`, `Runner` | `agents` | Core primitives: define agents, configure model settings, execute runs |
| `function_tool` | `agents` | Decorator used in Cell 8 to give the sub-agent a real callable tool |
| `ItemHelpers` | `agents` | Extracts text from `MessageOutputItem` entries in `new_items` |
| `ToolCallItem` | `agents` | Represents the orchestrator calling a tool — carries `.tool_name` and `.arguments` |
| `ToolCallOutputItem` | `agents` | Represents what a tool returned — carries `.output` |
| `MessageOutputItem` | `agents` | Represents an LLM text reply in `new_items` |
| `RunResult` | `agents.result` | Type annotation for `custom_output_extractor` — not re-exported from the top-level `agents` namespace |
| `BaseModel`, `Field` | `pydantic` | Define the structured input schema for `parameters=` in Cell 9 |
| `Reasoning` | `openai.types.shared` | Configures reasoning effort on GPT-5 models (`effort="none"` for low latency) |
| `json` | stdlib | Pretty-printing JSON schemas in inspection cells |

**Note:** `RunResult` must be imported from `agents.result`. Everything else in this table comes from the top-level `agents` namespace.

In [ ]:
from agents import (
    Agent,
    ItemHelpers,
    MessageOutputItem,
    ModelSettings,
    Runner,
    ToolCallItem,
    ToolCallOutputItem,
    function_tool,
)
from agents.result import RunResult
from pydantic import BaseModel, Field
from openai.types.shared import Reasoning
import json

## Concept: `as_tool()` vs Handoffs

Before writing any code, lock in the mental model. The SDK docstring for `Agent.as_tool()` states two fundamental differences from handoffs.

### Difference 1 — What the sub-agent receives

| Pattern | What the sub-agent gets |
|---|---|
| **Handoff** | The full conversation history up to the point of the handoff |
| **`as_tool()`** | A generated string or structured payload only. Not the conversation history. |

### Difference 2 — Who controls the conversation after

| Pattern | Control |
|---|---|
| **Handoff** | The receiving agent takes over the conversation entirely |
| **`as_tool()`** | The calling orchestrator retains control. The sub-agent is called like a function, returns a string, and the orchestrator continues. |

### What always happens to `result.final_output`

In the `as_tool()` pattern, `result.final_output` is **always** the orchestrator's own synthesis. Here is why: the sub-agent finishes its nested run, the SDK extracts a string from it (via the cascade below), and that string reaches the orchestrator's LLM as a tool call result. The orchestrator then writes its own reply on top of all tool results it collected. That reply is `result.final_output`.

The output extraction cascade (from `src/agents/agent.py`):
```
1. custom_output_extractor set? → call it and return immediately
2. run_result.final_output not None and not empty? → return it
3. Scan new_items in reverse:
   - MessageOutputItem with text → return that text
   - ToolCallOutputItem with string → return that output
4. Fallback: return run_result.final_output
```

`custom_output_extractor` does not change `result.final_output` directly. It changes **what the orchestrator's LLM reads as the tool result** — which then shapes what the orchestrator synthesises.

### Practical rule

- **Use `as_tool()`** when the orchestrator needs results back from specialists and must stay in charge throughout.
- **Use handoffs** when a specialist should own the rest of the conversation end-to-end.

> Handoffs are covered in depth in Section 5. This lecture focuses exclusively on `as_tool()`.

## `as_tool()` Parameter Reference

The full method signature is broad. Here are the parameters covered in this lecture:

| Parameter | Type | Default | What it does |
|---|---|---|---|
| `tool_name` | `str \| None` | Agent name (snake-cased) | The name the orchestrator LLM sees when deciding whether to call this tool |
| `tool_description` | `str \| None` | `""` | Natural-language description guiding the orchestrator on when to call |
| `custom_output_extractor` | `async (RunResult) -> str \| None` | `None` | Override which string is returned from the nested run to the orchestrator |
| `is_enabled` | `bool \| callable` | `True` | Toggle tool visibility at runtime without changing the orchestrator definition |
| `on_stream` | `callable \| None` | `None` | Receive streaming events from the nested run (Section 4) |
| `max_turns` | `int \| None` | `DEFAULT_MAX_TURNS` | Max LLM turns allowed in the nested sub-agent run |
| `run_config` | `RunConfig \| None` | `None` | Pass a `RunConfig` to the nested run |
| `failure_error_function` | `callable \| None` | default handler | Called when the sub-agent run fails; returns an error message to the orchestrator |
| `needs_approval` | `bool \| callable` | `False` | Human-in-the-loop approval gate (Update Section U3) |
| `parameters` | `type \| None` | `None` | Pydantic `BaseModel` or dataclass for structured input instead of the default single string |
| `include_input_schema` | `bool` | `False` | Include the full JSON Schema in the sub-agent's input text when `parameters` is set |

**Default input schema:** When `parameters=None`, the SDK uses `AgentAsToolInput` from `agents/agent_tool_input.py` — a Pydantic model with one field, `input: str`. We inspect this in Cell 6.

## Cell 5 — Basic `as_tool()`: Translation Orchestrator

The simplest form of manager-style orchestration: two specialist agents registered as tools on a central orchestrator. The orchestrator calls both to fulfil a single request and synthesises the combined result.

**What to observe:**
- The return value of `spanish_agent.as_tool(...)` drops directly into the `tools=` list — the same as any `@function_tool`.
- `tool_name` and `tool_description` are the only required arguments. Everything else uses defaults.
- Neither sub-agent sees the orchestrator's conversation. Each receives only the string the orchestrator decides to pass.
- `result.final_output` is the orchestrator's own synthesis over both tool results — not either sub-agent's reply directly.

In [ ]:
spanish_agent = Agent(
    name="Spanish Agent",
    instructions="You translate the user's message to Spanish.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

french_agent = Agent(
    name="French Agent",
    instructions="You translate the user's message to French.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

orchestrator = Agent(
    name="Translation Orchestrator",
    instructions=(
        "You are a translation orchestrator. "
        "Use the available tools to translate text as requested. "
        "If multiple translations are needed, call each tool."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    #TODO: register spanish_agent and french_agent as tools
)

result = await Runner.run(
    orchestrator,
    "Translate 'Good morning, how are you?' to both Spanish and French.",
)
print(result.final_output)

Spanish: Buenos días, ¿cómo estás?  
French: Bonjour, comment ça va ?


## Cell 6 — Inspecting the `FunctionTool` Returned by `as_tool()`

This cell creates a tool object without running any agent and examines its key properties.

| Attribute | Expected value | Why it matters |
|---|---|---|
| `type(spanish_tool)` | `FunctionTool` | Confirms `as_tool()` returns the same type as `@function_tool` — no special handling needed |
| `spanish_tool.name` | `"translate_to_spanish"` | The exact string the orchestrator LLM uses when deciding to call this tool |
| `_is_agent_tool` | `True` | Internal SDK flag for tracing and approval flows |
| `params_json_schema` | `{"properties": {"input": {"type": "string"}}, ...}` | The **default schema** — one required `input` field of type `string` |

The sub-agent receives only whatever the orchestrator puts in that `input` field. It has no visibility into the broader conversation. Keep this default schema in mind — Cell 9 replaces it with a three-field structured schema and you will compare the two directly.

In [ ]:
spanish_tool = spanish_agent.as_tool(
    tool_name="translate_to_spanish",
    tool_description="Translate the user's message to Spanish.",
)

print("Tool type:    ", type(spanish_tool))
print("Tool name:    ", spanish_tool.name)
print("Is agent tool:", getattr(spanish_tool, "_is_agent_tool", False))
print("\nDefault JSON schema:")
print(json.dumps(spanish_tool.params_json_schema, indent=2))

Tool type:     <class 'agents.tool.FunctionTool'>
Tool name:     translate_to_spanish
Is agent tool: True

Default JSON schema:
{
  "description": "Default input schema for agent-as-tool calls.",
  "properties": {
    "input": {
      "title": "Input",
      "type": "string"
    }
  },
  "required": [
    "input"
  ],
  "title": "AgentAsToolInput",
  "type": "object",
  "additionalProperties": false
}


## Cell 7 — Controlling Nested Run Depth with `max_turns`

Each call to `as_tool()` launches an **independent nested run** for that sub-agent. `max_turns` controls how many LLM turns that nested run is allowed.

**Key facts:**
- **Nested runs are completely independent.** The outer orchestrator's turn counter is separate from each sub-agent's counter. A sub-agent hitting its limit has no effect on the outer run.
- **The default is `DEFAULT_MAX_TURNS`** (10 in current SDK versions). Reduce it for simple single-step tasks; raise it for agents that need multiple internal tool calls.
- **The orchestrator only ever sees the final output string.** Everything that happens inside the nested run is invisible to it — intermediate steps, internal tool calls, all of it.

In this example, `research_agent` gets up to 3 turns and `summariser_agent` gets up to 2 turns. Both run independently and the orchestrator synthesises their results.

In [ ]:
research_agent = Agent(
    name="Research Agent",
    instructions=(
        "You are a research specialist. "
        "Answer questions thoroughly using your knowledge."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

summariser_agent = Agent(
    name="Summariser Agent",
    instructions=(
        "You are a summarisation specialist. "
        "Create concise, bullet-point summaries."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

orchestrator2 = Agent(
    name="Research Orchestrator",
    instructions=(
        "You are a research and summarisation orchestrator. "
        "First research the topic, then summarise the findings."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[
        research_agent.as_tool(
            tool_name="research_topic",
            tool_description="Research a topic and return detailed information.",
            max_turns=3,
        ),
        summariser_agent.as_tool(
            tool_name="summarise_content",
            tool_description="Summarise the provided content into bullet points.",
            max_turns=2,
        ),
    ],
)

result = await Runner.run(
    orchestrator2,
    "Research the history of the internet and summarise it.",
)
print(result.final_output)

- **1960s:** Packet switching and ARPA-funded research created the technical foundation.
- **1969–1970s:** **ARPANET** proved networking worked; **TCP/IP** let separate networks connect.
- **1980s:** TCP/IP became standard; **DNS** and **NSFNET** helped the internet scale.
- **1990s:** The **World Wide Web** made it public and easy to use; browsers drove mass adoption.
- **2000s:** Commercial growth, the dot-com boom, search, e-commerce, and social media expanded its role.
- **2010s:** Smartphones, apps, and cloud computing made the internet always-on and mobile.
- **2020s:** Privacy, security, regulation, and AI became major themes.

**Key turning points:** packet switching, ARPANET, TCP/IP, DNS, the Web, commercialization, mobile/cloud.


## Cell 8 — `custom_output_extractor`: Intercepting Raw Tool Output

For a sub-agent with **no tools**, `new_items` only ever contains `MessageOutputItem` entries — the LLM's text replies. No `ToolCallOutputItem` is ever produced, so there is nothing raw to extract. To demonstrate the extractor meaningfully, the sub-agent in this cell has a **real `@function_tool`** that returns JSON from a local product catalogue.

### The sub-agent's nested run produces this sequence

| Item in `new_items` | What it represents |
|---|---|
| `ToolCallItem` | The LLM decided to call `fetch_product_info` |
| `ToolCallOutputItem` | The raw JSON string returned by the function |
| `MessageOutputItem` | The LLM's synthesised prose reply describing the product |

### Two versions of the same tool

We register `data_agent` as a tool twice:
- `get_product_data_default` — no extractor. The orchestrator receives the sub-agent's synthesised prose (the `MessageOutputItem` text).
- `get_product_data_raw` — with an extractor that hunts for the `ToolCallOutputItem` and returns its `.output`. The orchestrator receives the raw JSON string.

### What `custom_output_extractor` actually changes

It changes **what string the orchestrator's LLM reads as the tool call result** — not `result.final_output` directly. `result.final_output` is always the orchestrator's own synthesis. But if the orchestrator reads raw JSON instead of synthesised prose, its synthesis is built on different information — more precise, no double-summarisation.

The `new_items` printout shows both `ToolCallOutputItem` entries at the orchestrator level side by side, so you can compare exactly what the orchestrator received from each version of the tool.

In [ ]:
# ── Step 1: a real tool the sub-agent will call ───────────────────────────────

@function_tool
def fetch_product_info(product_id: str) -> str:
    """Return raw product data for a given product ID."""
    catalogue = {
        "P001": '{"id": "P001", "name": "Wireless Headphones", "price": 79.99, "stock": 142}',
        "P002": '{"id": "P002", "name": "USB-C Hub",           "price": 34.99, "stock": 0}',
        "P003": '{"id": "P003", "name": "Mechanical Keyboard", "price": 129.99, "stock": 57}',
    }
    return catalogue.get(product_id, '{"error": "Product not found"}')


# ── Step 2: sub-agent WITH a real tool ───────────────────────────────────────

data_agent = Agent(
    name="Data Agent",
    instructions=(
        "You are a product data specialist. "
        "When asked about a product, call fetch_product_info with its ID, "
        "then write a friendly one-sentence summary of the result."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[fetch_product_info],
)


# ── Step 3: extractor — find the ToolCallOutputItem, skip the synthesis ───────

async def extract_raw_tool_output(run_result: RunResult) -> str:
    """Return the raw tool call output from the nested run, bypassing LLM synthesis."""
    # TODO: write an async extractor that scans run_result.new_items in reverse,
    # finds the last ToolCallOutputItem with a string output, and returns it.
    # Fall back to run_result.final_output if nothing is found.


# ── Step 4: two versions of the tool ─────────────────────────────────────────

# Orchestrator receives the sub-agent's synthesised prose
# TODO: register data_agent as a tool twice:
# - get_product_data_default: no extractor (orchestrator gets synthesised prose)
# - get_product_data_raw: with custom_output_extractor=extract_raw_tool_output


# ── Step 5: orchestrator calls both tools for the same product ────────────────

orchestrator3 = Agent(
    name="Product Orchestrator",
    instructions=(
        "You are a product information orchestrator. "
        "When asked about a product, call BOTH tools with the same product ID "
        "and report what each one returned, clearly labelled."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[data_tool_default, data_tool_raw],
)

result = await Runner.run(
    orchestrator3,
    "Look up product P001 using both tools and tell me what each one returned.",
)

# ── Step 6: inspect orchestrator new_items to see what each tool returned ─────

print("=== Orchestrator new_items ===")
for i, item in enumerate(result.new_items):
    if isinstance(item, ToolCallItem):
        print(f"  [{i}] ToolCallItem       tool={item.tool_name}")
    elif isinstance(item, ToolCallOutputItem):
        print(f"  [{i}] ToolCallOutputItem output={str(item.output)[:120]}")
    elif isinstance(item, MessageOutputItem):
        text = ItemHelpers.text_message_output(item)
        if text:
            print(f"  [{i}] MessageOutputItem  text={text[:80]}")
    else:
        print(f"  [{i}] {type(item).__name__}")

print("\n=== result.final_output (orchestrator synthesis) ===")
print(result.final_output)

=== Orchestrator new_items ===
  [0] ToolCallItem       tool=get_product_data_default
  [1] ToolCallItem       tool=get_product_data_raw
  [2] ToolCallOutputItem output=Wireless Headphones (P001) are priced at $79.99 and currently in stock with 142 units available.
  [3] ToolCallOutputItem output={"id": "P001", "name": "Wireless Headphones", "price": 79.99, "stock": 142}
  [4] MessageOutputItem  text=Default tool returned: Wireless Headphones (P001) are priced at $79.99 and curre

=== result.final_output (orchestrator synthesis) ===
Default tool returned: Wireless Headphones (P001) are priced at $79.99 and currently in stock with 142 units available.

Raw tool returned: {"id":"P001","name":"Wireless Headphones","price":79.99,"stock":142}


## Cell 9 — Structured Input with `parameters=`

By default, the orchestrator passes a single string in the `input` field. For well-defined inter-agent contracts — where the sub-agent needs multiple distinct pieces of information — a structured schema forces the orchestrator to provide each piece explicitly and lets the SDK validate them before the sub-agent ever runs.

### What happens end-to-end

| Step | What happens |
|---|---|
| Schema exposed | The orchestrator sees the Pydantic model's JSON schema instead of `{"input": "string"}` |
| Orchestrator fills it in | The LLM emits a structured JSON call with all required fields populated |
| SDK validates | The JSON is parsed and validated against the Pydantic model — a `ModelBehaviorError` is raised if invalid |
| Sub-agent receives | The SDK calls `default_tool_input_builder`, which wraps the validated fields into a formatted text block with the preamble: *"You are being called as a tool. The following is structured input data..."* |
| Sub-agent replies | Plain text, returned to the orchestrator as the tool result string |
| Orchestrator synthesises | `result.final_output` is always the orchestrator's own reply — always synthesised |

The sub-agent never sees a Pydantic object. It reads the formatted text block the SDK builds from the validated fields.

### What this cell makes visible

- The JSON schema printed at the top shows three explicit fields instead of Cell 6's single `input` field.
- The `ToolCallItem.arguments` shows the exact structured JSON the orchestrator's LLM filled in and passed — not a free-form string, but a validated typed payload.
- `result.final_output` is the orchestrator's synthesis, now built on a precise, contract-driven call.

**Constraint:** `parameters=` must receive a Pydantic `BaseModel` subclass or a `dataclass`. Any other type raises a `TypeError` at call time, verified from `_is_supported_parameters()` in `src/agents/agent.py`.

In [ ]:
# ── Step 1: define the structured input schema ────────────────────────────────

class TranslationInput(BaseModel):
    text: str = Field(description="The text to translate.")
    source_language: str = Field(description="The language to translate from, e.g. 'English'.")
    target_language: str = Field(description="The language to translate into, e.g. 'French'.")


# ── Step 2: sub-agent that translates structured input ────────────────────────

translator_agent = Agent(
    name="Translator Agent",
    instructions=(
        "You are a professional translator. "
        "You will receive structured input specifying text, source language, and target language. "
        "Translate the text and return only the translation, nothing else."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)


# ── Step 3: register with structured parameters ───────────────────────────────

translator_tool = translator_agent.as_tool(
    tool_name="translate_text",
    tool_description=(
        "Translate text from one language to another. "
        "Provide the text, the source language, and the target language."
    ),
    # TODO: pass TranslationInput as the parameters argument
)

print("Schema the orchestrator sees (compare to Cell 6 default schema):")
print(json.dumps(translator_tool.params_json_schema, indent=2))


# ── Step 4: orchestrator calls the tool ──────────────────────────────────────

orchestrator4 = Agent(
    name="Translation Coordinator",
    instructions=(
        "You are a translation coordinator. "
        "Use the translate_text tool to fulfil translation requests. "
        "Always call the tool — never translate yourself."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[translator_tool],
)

result = await Runner.run(
    orchestrator4,
    "Please translate 'Hello, how are you?' from English to French.",
)

# ── Step 5: show the structured JSON the orchestrator passed to the tool ──────

print("\n=== What the orchestrator passed to the tool ===")
for item in result.new_items:
    if isinstance(item, ToolCallItem):
        print(f"Tool called : {item.tool_name}")
        arguments = getattr(item.raw_item, "arguments", None)
        print(f"Arguments   : {arguments}")

print("\n=== result.final_output ===")
print(result.final_output)

Schema the orchestrator sees (compare to Cell 6 default schema):
{
  "properties": {
    "text": {
      "description": "The text to translate.",
      "title": "Text",
      "type": "string"
    },
    "source_language": {
      "description": "The language to translate from, e.g. 'English'.",
      "title": "Source Language",
      "type": "string"
    },
    "target_language": {
      "description": "The language to translate into, e.g. 'French'.",
      "title": "Target Language",
      "type": "string"
    }
  },
  "required": [
    "text",
    "source_language",
    "target_language"
  ],
  "title": "TranslationInput",
  "type": "object",
  "additionalProperties": false
}

=== What the orchestrator passed to the tool ===
Tool called : translate_text
Arguments   : {"text":"Hello, how are you?","source_language":"English","target_language":"French"}

=== result.final_output ===
Bonjour, comment ça va ?


## Summary: Manager Pattern vs Handoff Pattern

| Dimension | `as_tool()` — Manager Pattern | Handoff Pattern |
|---|---|---|
| **Who controls the conversation?** | Orchestrator retains control throughout | Receiving agent takes over entirely |
| **What does the sub-agent receive?** | Generated string or structured payload only | Full conversation history |
| **Is `result.final_output` synthesised?** | Always — orchestrator LLM writes its own reply over all tool results | Always — receiving agent writes the final reply |
| **What does `custom_output_extractor` change?** | The string the orchestrator reads as the tool result — not `result.final_output` directly | N/A |
| **Can combine multiple specialist results?** | Yes — orchestrator synthesises from all tool calls | No — one specialist handles end-to-end |
| **Best for** | Parallel tasks, data extraction, multi-step synthesis | Customer support routing, escalation, domain takeover |

Handoffs are covered in depth in **Section 5**.